# Session 6: PINN for the 1D heat equation

We now apply the physics-informed neural network (PINN) framework (introduced in [Session 5](Session5.ipynb)) to our first full PDE: the **1D heat equation**. This is an ideal starting point because:
- It is a parabolic PDE with both a spatial second derivative and a time derivative.
- An exact analytical solution exists for simple boundary and initial conditions, so we can rigorously verify our PINN.
- The physics is intuitive — temperature diffusing along a rod.

## 1. Problem statement

We solve:

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}, \quad x \in [0, 1],\; t \in [0, 1]
$$

with:
- **Boundary conditions** (Dirichlet): $u(0, t) = u(1, t) = 0$
- **Initial condition**: $u(x, 0) = \sin(\pi x)$
- **Thermal diffusivity**: $\alpha = 0.01$

### Exact solution

By separation of variables, the exact solution is:

$$
u(x, t) = e^{-\alpha \pi^2 t} \sin(\pi x)
$$

The amplitude decays exponentially in time while the spatial shape remains a sine wave. This is easy to verify: $u_t = -\alpha\pi^2 e^{-\alpha\pi^2 t}\sin(\pi x)$ and $u_{xx} = -\pi^2 e^{-\alpha\pi^2 t}\sin(\pi x)$, so $u_t = \alpha u_{xx}$. ✓

## 2. The PINN architecture

The network takes the 2D input $(x, t)$ and outputs the scalar field $u_\theta(x, t)$.

$$
u_\theta: \mathbb{R}^2 \to \mathbb{R}
$$

We use tanh activations throughout and 4 hidden layers of width 50, which is more than sufficient for this smooth solution.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class PINN(nn.Module):
    """MLP with tanh activations. Input: (x, t), Output: u."""
    def __init__(self, layers=[2, 50, 50, 50, 50, 1]):
        super().__init__()
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i+1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, X):
        return self.net(X)

## 3. Sampling training points

We sample three sets of points:
- **Collocation points** $(x_i, t_i) \in (0,1)^2$: interior points where the PDE must hold.
- **Boundary points** $(0, t_j)$ and $(1, t_j)$: where $u = 0$.
- **Initial condition (IC) points** $(x_k, 0)$: where $u = \sin(\pi x_k)$.

In [ ]:
alpha = 0.01
N_colloc = 10000
N_bc     = 200
N_ic     = 200

torch.manual_seed(42)

# --- Collocation (interior): requires_grad for autograd ---
colloc = torch.rand(N_colloc, 2, requires_grad=True, device=device)

# --- Boundary: x=0 and x=1 ---
t_bc = torch.rand(N_bc, device=device)
bc_left  = torch.stack([torch.zeros(N_bc, device=device),  t_bc], dim=1)
bc_right = torch.stack([torch.ones(N_bc,  device=device),  t_bc], dim=1)
bc = torch.cat([bc_left, bc_right], dim=0)       # shape (2*N_bc, 2)

# --- Initial condition: t=0 ---
x_ic = torch.rand(N_ic, device=device)
ic = torch.stack([x_ic, torch.zeros(N_ic, device=device)], dim=1)
u_ic_true = torch.sin(np.pi * x_ic).reshape(-1, 1)

## 4. Loss function

The three loss components are:

$$
\mathcal{L}_{\text{PDE}} = \frac{1}{N_f} \sum_i \left( \frac{\partial u_\theta}{\partial t} - \alpha \frac{\partial^2 u_\theta}{\partial x^2} \right)^2_{(x_i, t_i)}
$$

$$
\mathcal{L}_{\text{BC}} = \frac{1}{2N_b} \sum_j u_\theta(0, t_j)^2 + u_\theta(1, t_j)^2
$$

$$
\mathcal{L}_{\text{IC}} = \frac{1}{N_0} \sum_k \left( u_\theta(x_k, 0) - \sin(\pi x_k) \right)^2
$$

The spatial derivative $u_{xx}$ requires two applications of `torch.autograd.grad` with `create_graph=True`.

In [ ]:
model = PINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def compute_pde_residual(model, pts):
    """Compute heat equation residual u_t - alpha * u_xx at pts."""
    u = model(pts)
    grads = torch.autograd.grad(
        u, pts,
        grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]
    u_x = grads[:, 0:1]
    u_t = grads[:, 1:2]
    u_xx = torch.autograd.grad(
        u_x, pts,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]
    return u_t - alpha * u_xx

## 5. Training

In [ ]:
epochs = 10000
loss_history = {'total': [], 'pde': [], 'bc': [], 'ic': []}

for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE residual
    residual = compute_pde_residual(model, colloc)
    loss_pde = torch.mean(residual**2)

    # Boundary conditions
    loss_bc = torch.mean(model(bc)**2)

    # Initial condition
    loss_ic = torch.mean((model(ic) - u_ic_true)**2)

    loss = loss_pde + loss_bc + loss_ic
    loss.backward()
    optimizer.step()

    loss_history['total'].append(loss.item())
    loss_history['pde'].append(loss_pde.item())
    loss_history['bc'].append(loss_bc.item())
    loss_history['ic'].append(loss_ic.item())

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:6d} | Total: {loss.item():.2e} | "
              f"PDE: {loss_pde.item():.2e} | BC: {loss_bc.item():.2e} | IC: {loss_ic.item():.2e}")

## 6. Results

### 6.1 Loss history

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for key, vals in loss_history.items():
    ax.semilogy(vals, label=key)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Components')
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Full solution field

In [ ]:
with torch.no_grad():
    Nx, Nt = 100, 100
    x_grid = torch.linspace(0, 1, Nx)
    t_grid = torch.linspace(0, 1, Nt)
    X, T = torch.meshgrid(x_grid, t_grid, indexing='ij')
    XT = torch.stack([X.flatten(), T.flatten()], dim=1).to(device)
    u_pred = model(XT).cpu().numpy().reshape(Nx, Nt)

u_exact_grid = (np.exp(-alpha * np.pi**2 * T.numpy()) * np.sin(np.pi * X.numpy()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].contourf(T.numpy(), X.numpy(), u_pred, levels=50, cmap='viridis')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('PINN Solution $u_\\theta(x,t)$')
axes[0].set_xlabel('t'); axes[0].set_ylabel('x')

im1 = axes[1].contourf(T.numpy(), X.numpy(), u_exact_grid, levels=50, cmap='viridis')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Exact Solution $u(x,t)$')
axes[1].set_xlabel('t'); axes[1].set_ylabel('x')

error = np.abs(u_pred - u_exact_grid)
im2 = axes[2].contourf(T.numpy(), X.numpy(), error, levels=50, cmap='hot_r')
plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('t'); axes[2].set_ylabel('x')

plt.tight_layout()
plt.show()

### 6.3 Slice comparison at selected times

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, t_val in zip(axes, [0.0, 0.5, 1.0]):
    x_line = torch.linspace(0, 1, 200).reshape(-1, 1)
    t_line = torch.full_like(x_line, t_val)
    xt_line = torch.cat([x_line, t_line], dim=1).to(device)

    with torch.no_grad():
        u_pinn = model(xt_line).cpu().numpy().flatten()
    u_an = np.exp(-alpha * np.pi**2 * t_val) * np.sin(np.pi * x_line.numpy().flatten())

    ax.plot(x_line.numpy(), u_an, 'k-', lw=2, label='Exact')
    ax.plot(x_line.numpy(), u_pinn, 'r--', lw=2, label='PINN')
    ax.set_title(f't = {t_val}')
    ax.set_xlabel('x'); ax.set_ylabel('u')
    ax.legend()

plt.suptitle('PINN vs. Exact Solution at Fixed Times', fontsize=13)
plt.tight_layout()
plt.show()

# Relative L2 error over the full grid
rel_l2 = np.sqrt(np.mean((u_pred - u_exact_grid)**2)) / np.sqrt(np.mean(u_exact_grid**2))
print(f'Relative L2 error: {rel_l2:.4e}')

## 7. Discussion

A few things to notice:

- **No grid required**: the PINN solves the PDE on a continuous domain. Evaluating at any $(x, t)$ pair is a simple forward pass.
- **Equal weight assumption**: we used $\lambda_{\text{PDE}} = \lambda_{\text{BC}} = \lambda_{\text{IC}} = 1$. In practice, these weights often need tuning — the losses may have very different magnitudes. [Session 8](Session8.ipynb) covers this.
- **Smooth solutions are easiest**: the heat equation has a very smooth solution, which plays to the strengths of neural networks with tanh activations. Sharper features (shocks, discontinuities) are harder — see [Session 7](Session7.ipynb) with Burgers' equation.

## 8. Summary and what comes next

We have successfully:
1. Formulated the heat equation PINN with all three loss components.
2. Computed $u_t$ and $u_{xx}$ via two levels of autograd.
3. Achieved good agreement with the analytical solution.

**[Session 7](Session7.ipynb)** tackles **Burgers' equation** — a non-linear PDE with potential shock formation. The PINN formulation is essentially the same, but the dynamics are richer and the training is more challenging.

**Reading**: Ben Moseley's tutorial [*So, what is a physics-informed neural network?*](https://benmoseley.blog/my-research/so-what-is-a-physics-informed-neural-network) provides an accessible companion to this notebook.

## Exercises

1. **Gaussian initial condition**: replace the $\sin(\pi x)$ initial condition with a Gaussian $u(x, 0) = \exp(-100(x - 0.5)^2)$ (keeping zero Dirichlet BCs). There is no simple closed-form solution, so compare the PINN result with a high-resolution explicit finite difference solution. How well does the PINN reproduce the diffusion?

2. **Loss weight sensitivity**: retrain the heat equation PINN with IC weight $\lambda_{\text{IC}} \in \{0.1, 1, 10, 100\}$ (keep PDE and BC weights at 1). Plot the relative $L_2$ error as a function of $\lambda_{\text{IC}}$ and identify the optimal range.

3. **Reduced collocation**: reduce the number of collocation points from 10,000 to 500 and retrain. At what point does the PDE residual become insufficiently enforced and the solution quality degrade noticeably?

4. **Higher diffusivity**: increase $\alpha$ from 0.01 to 0.1. The solution now decays much faster in time. Does the PINN need more epochs, a different loss weight balance, or a deeper network to achieve a comparable relative $L_2$ error?